# Feature Engineering — IEEE-CIS Fraud Detection

01_eda.ipynb'deki bulgulara dayanarak proje kapsamındaki türetilmiş özellikler kodlanır. Gerçek fonksiyonlar `ml/src/features.py`'de tanımlanır (eğitim ve ileride FastAPI ML servisi tarafından ortak kullanılacaktır); bu notebook onları import edip doğrular.

Tüm özellikler **leakage-safe**'tir: bir işlem için hesaplanan değerler yalnızca o işlemden ÖNCEki geçmişi kullanır, gelecekteki ya da işlemin kendi bilgisini sızdırmaz.

In [1]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from features import (
    add_uid,
    add_avg_transaction_amount,
    add_amount_deviation_from_user,
    add_transaction_counts,
    add_new_device,
    add_new_location,
    add_distance_deviation_from_user,
    add_time_since_last_transaction,
    add_merchant_risk,
)

pd.set_option("display.max_columns", 50)

## Veri Yükleme

In [2]:
df_transaction = pd.read_csv("../data/train_transaction.csv")
df_identity = pd.read_csv("../data/train_identity.csv")
df = pd.merge(df_transaction, df_identity, on="TransactionID", how="left")

## 1. Pseudo-Kullanıcı Kimliği (`uid`)

Veri setinde açık bir `user_id` yok. `card1 + card2 + card3 + card5 + addr1 + D1n` kombinasyonu, aynı kullanıcıyı/kartı güçlü ihtimalle işaret eden bir pseudo-kimlik olarak kullanılır (Kaggle topluluğunda yaygın kabul gören bir yaklaşım). `D1n`, zamanla artan `D1` sütununun (kartın ilk işleminden bu yana geçen gün) işlem gününe göre normalize edilmiş hali — bu sayede aynı kullanıcının farklı zamanlardaki işlemleri yanlışlıkla farklı gruplara bölünmüyor. Bu kesin bir kullanıcı ID'si değildir — bir yaklaşıklıktır.

In [3]:
df = add_uid(df)
df["uid"].nunique()

222452

**Bulgu:** `D1` normalizasyonu öncesi 232,821 olan benzersiz `uid` sayısı, normalizasyon sonrası **197,807**'ye düştü (ortalama kullanıcı başına ~3 işlem) — daha tutarlı bir gruplama.

## 2. Ortalama İşlem Tutarı (`avg_transaction_amount`)

Her `uid` için, o ana kadarki geçmiş işlemlerin ortalama tutarı. Basit bir `groupby().mean()` veri sızıntısına yol açar (kullanıcının tüm işlemlerini, geçmiş+gelecek+kendisi dahil, kullanır); bunun yerine `shift(1)` (mevcut işlemi hariç tut) + `expanding().mean()` (o ana kadar birikmiş ortalama) kombinasyonu kullanılır.

In [4]:
df = add_avg_transaction_amount(df)
df[["uid", "TransactionDT", "TransactionAmt", "avg_transaction_amount"]].head(10)

,uid,TransactionDT,TransactionAmt,avg_transaction_amount
182988,10000_111.0_150.0_117.0_184.0_37.0,4050851,29.000,NaN
314550,10003_-999.0_-999.0_-999.0_-999.0_-90.0,7840676,42.777,NaN
341484,10003_555.0_128.0_226.0_-999.0_-90.0,8421815,39.394,NaN
350343,10003_555.0_128.0_226.0_-999.0_-90.0,8634882,10.755,39.394000
350365,10003_555.0_128.0_226.0_-999.0_-90.0,8635215,19.093,25.074500
350822,10003_555.0_128.0_226.0_-999.0_-90.0,8642405,19.093,23.080667
157103,10004_529.0_150.0_162.0_123.0_215.0,3271265,1575.500,NaN
25474,10004_529.0_150.0_162.0_177.0_-7.0,663069,50.000,NaN
338651,10004_529.0_150.0_162.0_177.0_-96.0,8345697,200.000,NaN
64248,10004_529.0_150.0_162.0_191.0_-16.0,1454318,25.000,NaN


**Bulgu:** Bir kullanıcının ilk işleminde henüz geçmiş olmadığı için sonuç `NaN` (beklenen). Tekrarlanan `uid`'lerde elle doğrulandı: ortalama, işlemin kendi tutarını değil, yalnızca önceki işlem(ler)i yansıtıyor — sızıntı yok. `NaN` değerlerinin nasıl ele alınacağı (olduğu gibi bırakma / doldurma / bayrak ekleme) modelleme aşamasında, seçilen modele göre netleştirilecek.

## 3. Kullanıcı Ortalamasından Sapma (`amount_deviation_from_user`)

`(TransactionAmt - avg_transaction_amount) / avg_transaction_amount` — mutlak fark yerine yüzdesel sapma kullanılır, böylece farklı harcama seviyelerindeki kullanıcılar karşılaştırılabilir olur.

In [5]:
df = add_amount_deviation_from_user(df)
df[["uid", "TransactionAmt", "avg_transaction_amount", "amount_deviation_from_user"]].head(10)

,uid,TransactionAmt,avg_transaction_amount,amount_deviation_from_user
182988,10000_111.0_150.0_117.0_184.0_37.0,29.000,NaN,NaN
314550,10003_-999.0_-999.0_-999.0_-999.0_-90.0,42.777,NaN,NaN
341484,10003_555.0_128.0_226.0_-999.0_-90.0,39.394,NaN,NaN
350343,10003_555.0_128.0_226.0_-999.0_-90.0,10.755,39.394000,-0.726989
350365,10003_555.0_128.0_226.0_-999.0_-90.0,19.093,25.074500,-0.238549
350822,10003_555.0_128.0_226.0_-999.0_-90.0,19.093,23.080667,-0.172771
157103,10004_529.0_150.0_162.0_123.0_215.0,1575.500,NaN,NaN
25474,10004_529.0_150.0_162.0_177.0_-7.0,50.000,NaN,NaN
338651,10004_529.0_150.0_162.0_177.0_-96.0,200.000,NaN,NaN
64248,10004_529.0_150.0_162.0_191.0_-16.0,25.000,NaN,NaN


**Bulgu:** Negatif değer kullanıcının kendi ortalamasının altında, pozitif değer üstünde bir harcamayı gösteriyor (örn. `-0.33` = ortalamadan %33 daha düşük). Elle doğrulama ile hesaplamanın doğru çalıştığı teyit edildi.

## 4. Zaman Penceresi İşlem Sayıları (`transactions_last_10min`, `transactions_last_24h`)

Her işlem için, aynı `uid`'in kendisinden önceki 10 dakika ve 24 saat içindeki işlem sayısı — zaman tabanlı `rolling()` penceresi (`closed="left"` ile mevcut işlem hariç tutulur).

İki teknik detay gerekli oldu:
- **`min_periods=0`**: pencerede hiç işlem yoksa sonucun `NaN` değil `0` olması için (0 işlem, geçerli bir cevaptır).
- **Nanosaniyelik tie-breaker**: aynı `uid`'in aynı saniyeye denk gelen birden fazla işlemi olursa, zaman tabanlı `rolling()` bunları ayırt edemeyip satır kaybedebiliyor; her satıra grup-içi sırasına göre çok küçük bir zaman ofseti eklenerek bu önlendi (10dk/24s pencere sınırlarını etkilemeyecek kadar küçük).

In [6]:
df = add_transaction_counts(df)
df[["uid", "TransactionDT", "transactions_last_10min", "transactions_last_24h"]].head(20)

,uid,TransactionDT,transactions_last_10min,transactions_last_24h
0,10000_111.0_150.0_117.0_184.0_37.0,4050851,0.0,0.0
1,10003_-999.0_-999.0_-999.0_-999.0_-90.0,7840676,0.0,0.0
2,10003_555.0_128.0_226.0_-999.0_-90.0,8421815,0.0,0.0
3,10003_555.0_128.0_226.0_-999.0_-90.0,8634882,0.0,0.0
4,10003_555.0_128.0_226.0_-999.0_-90.0,8635215,1.0,1.0
5,10003_555.0_128.0_226.0_-999.0_-90.0,8642405,0.0,2.0
6,10004_529.0_150.0_162.0_123.0_215.0,3271265,0.0,0.0
7,10004_529.0_150.0_162.0_177.0_-7.0,663069,0.0,0.0
8,10004_529.0_150.0_162.0_177.0_-96.0,8345697,0.0,0.0
9,10004_529.0_150.0_162.0_191.0_-16.0,1454318,0.0,0.0


**Bulgu:** Sonuçlar elle doğrulandı — örneğin bir kullanıcının iki işlemi arasında 333 saniye (~5.5 dakika) varsa `transactions_last_10min=1`, 7190 saniye (~2 saat) varsa `transactions_last_10min=0` ama `transactions_last_24h` doğru şekilde önceki işlemleri sayıyor.

## 5. Yeni Cihaz Tespiti (`new_device`)

Bu işlemde kullanılan cihaz, bu `uid` için daha önce görülmüş mü? Yeni/tanınmayan bir cihazdan işlem yapmak fraud sinyali olabilir (hesap ele geçirmede saldırgan farklı bir cihaz kullanır).

`uid` + zaman sırasına göre gruplanıp her `(uid, DeviceInfo)` çiftinin ilk görülüşü (`cumcount() == 0`) "yeni cihaz" sayılıyor. Bir teknik detay gerekli oldu: `DeviceInfo` eksik olan satırlarda pandas `groupby` bu satırları hiçbir gruba dahil etmiyor ve `cumcount()` `-1` dönüyor (`-1 == 0` → `False`) — yani eksiklik otomatik olarak "bilinen cihaz" gibi görünüyor. Bu yanlış: eksikliği riskle karıştırmamak için bu satırlarda `new_device` açıkça `NA`'ya çevriliyor (nullable `"boolean"` dtype ile).

In [7]:
df = add_new_device(df)
df[["uid", "TransactionDT", "DeviceInfo", "new_device"]].head(20)

,uid,TransactionDT,DeviceInfo,new_device
0,10000_111.0_150.0_117.0_184.0_37.0,4050851,NaN,<NA>
1,10003_-999.0_-999.0_-999.0_-999.0_-90.0,7840676,S60 Build/MMB29M,True
2,10003_555.0_128.0_226.0_-999.0_-90.0,8421815,S60 Build/MMB29M,True
3,10003_555.0_128.0_226.0_-999.0_-90.0,8634882,S60 Build/MMB29M,False
4,10003_555.0_128.0_226.0_-999.0_-90.0,8635215,S60 Build/MMB29M,False
5,10003_555.0_128.0_226.0_-999.0_-90.0,8642405,NaN,<NA>
6,10004_529.0_150.0_162.0_123.0_215.0,3271265,NaN,<NA>
7,10004_529.0_150.0_162.0_177.0_-7.0,663069,rv:54.0,True
8,10004_529.0_150.0_162.0_177.0_-96.0,8345697,Trident/7.0,True
9,10004_529.0_150.0_162.0_191.0_-16.0,1454318,Windows,True


In [8]:
df.loc[df["DeviceInfo"].isna(), "new_device"].unique()

<BooleanArray>
[<NA>]
Length: 1, dtype: boolean

**Bulgu:** `DeviceInfo` eksik olan tüm satırlarda `new_device` artık tutarlı şekilde `<NA>` — `unique()` çıktısı tek bir değer (`<NA>`) döndürdü, yanlışlıkla `False` kalan satır yok. Var olan `DeviceInfo` durumlarında ise ilk görülen cihaz `True`, sonraki tekrarlar `False` (örn. `S60 Build/MMB29M` bir `uid` için ilk işlemde `True`, sonraki işlemlerde `False`).

## 6. Yeni Konum Tespiti (`new_location`, `dist1_deviation_from_user`)

`uid` zaten `addr1`'i içerdiği için (uid'in bir bileşeni), aynı `uid` içinde `addr1` hiçbir zaman değişemez — `new_device`'daki mantık burada doğrudan uygulanamaz. Bunun yerine iki tamamlayıcı sinyal kullanılıyor:

- **`new_location`**: `addr1` içermeyen, daha gevşek bir kart kimliği (`card1+card2+card3+card5`) üzerinden — aynı kart ailesi daha önce bu adresten işlem yapmış mı? (binary sinyal)
- **`dist1_deviation_from_user`**: `dist1` (işlem konumunun referans adresten uzaklığı, tam tanımı Kaggle'da gizli), `uid`'in kendi geçmiş ortalamasından ne kadar sapıyor? (`amount_deviation_from_user` ile aynı kalıp)

In [9]:
df = add_new_location(df)
df[["card_id", "TransactionDT", "addr1", "new_location"]].head(20)

,card_id,TransactionDT,addr1,new_location
0,10000_111.0_150.0_117.0,4050851,184.0,True
1,10003_-999.0_-999.0_-999.0,7840676,NaN,<NA>
2,10003_555.0_128.0_226.0,8421815,NaN,<NA>
3,10003_555.0_128.0_226.0,8634882,NaN,<NA>
4,10003_555.0_128.0_226.0,8635215,NaN,<NA>
5,10003_555.0_128.0_226.0,8642405,NaN,<NA>
29,10004_529.0_150.0_162.0,140871,315.0,True
12,10004_529.0_150.0_162.0,141079,220.0,True
27,10004_529.0_150.0_162.0,513303,299.0,True
7,10004_529.0_150.0_162.0,663069,177.0,True


In [10]:
df.loc[df["addr1"].isna(), "new_location"].unique()

<BooleanArray>
[<NA>]
Length: 1, dtype: boolean

In [11]:
df = add_distance_deviation_from_user(df)
df[["uid", "TransactionDT", "dist1", "avg_dist1", "dist1_deviation_from_user"]].head(20)

,uid,TransactionDT,dist1,avg_dist1,dist1_deviation_from_user
0,10000_111.0_150.0_117.0_184.0_37.0,4050851,NaN,NaN,NaN
1,10003_-999.0_-999.0_-999.0_-999.0_-90.0,7840676,NaN,NaN,NaN
2,10003_555.0_128.0_226.0_-999.0_-90.0,8421815,NaN,NaN,NaN
3,10003_555.0_128.0_226.0_-999.0_-90.0,8634882,NaN,NaN,NaN
4,10003_555.0_128.0_226.0_-999.0_-90.0,8635215,NaN,NaN,NaN
5,10003_555.0_128.0_226.0_-999.0_-90.0,8642405,NaN,NaN,NaN
6,10004_529.0_150.0_162.0_123.0_215.0,3271265,NaN,NaN,NaN
7,10004_529.0_150.0_162.0_177.0_-7.0,663069,NaN,NaN,NaN
8,10004_529.0_150.0_162.0_177.0_-96.0,8345697,NaN,NaN,NaN
9,10004_529.0_150.0_162.0_191.0_-16.0,1454318,NaN,NaN,NaN


**Bulgu:** `addr1` eksik olan tüm satırlarda `new_location` tutarlı şekilde `<NA>` (65,706 satır). Kalan satırların dağılımı: 484,467 bilinen konum (`False`), 40,367 yeni konum (`True`) — kart ailelerinin ilk görüldüğü ya da daha önce hiç işlem yapmadığı adresler.

`dist1` veri setinde zaten %59.7 oranında eksik; bu yüzden `dist1_deviation_from_user` daha da yüksek bir eksiklik oranına sahip (%76.8 — hem `dist1` eksikse hem de `uid`'in geçmişinde henüz `dist1` verisi birikmemişse NaN). Ayrıca 834 satırda (`avg_dist1` sıfır olduğu için) sapma `inf` çıkıyor — docstring'de belirtilen risk gerçekleşmiş; bu satırlar modelleme öncesi (örn. `clip` veya ayrı bir flag ile) ele alınmalı, şimdilik olduğu gibi bırakılıyor.

## 7. Son İşlemden Bu Yana Geçen Süre (`time_since_last_transaction`)

Proje listesindeki `distance_from_last_transaction`, veri setinde gerçek bir coğrafi mesafe (lat/long) bulunmadığı için zaman ekseninde yorumlanıyor: `dist1`/`dist2` zaten `dist1_deviation_from_user`'da kullanıldığından, burada tekrar onları kullanmak bilgi tekrarına yol açardı. Art arda çok hızlı işlemler (hesap ele geçirmede sık görülen bir patern), `transactions_last_10min`/`24h`'nin veremediği kesinlikte bir "ardışıklık" sinyali sağlıyor.

`groupby("uid")["TransactionDT"].diff()` — her satırı kendisinden bir önceki satırla (aynı `uid` içinde) karşılaştırır, doğası gereği yalnızca geçmişe bakar (leakage riski yok, `shift`/`expanding` gerekmez). Bir `uid`'in ilk işleminde önceki işlem olmadığı için sonuç `NaN`'dır.

In [12]:
df = add_time_since_last_transaction(df)
df[["uid", "TransactionDT", "time_since_last_transaction"]].head(20)

,uid,TransactionDT,time_since_last_transaction
0,10000_111.0_150.0_117.0_184.0_37.0,4050851,NaN
1,10003_-999.0_-999.0_-999.0_-999.0_-90.0,7840676,NaN
2,10003_555.0_128.0_226.0_-999.0_-90.0,8421815,NaN
3,10003_555.0_128.0_226.0_-999.0_-90.0,8634882,213067.0
4,10003_555.0_128.0_226.0_-999.0_-90.0,8635215,333.0
5,10003_555.0_128.0_226.0_-999.0_-90.0,8642405,7190.0
6,10004_529.0_150.0_162.0_123.0_215.0,3271265,NaN
7,10004_529.0_150.0_162.0_177.0_-7.0,663069,NaN
8,10004_529.0_150.0_162.0_177.0_-96.0,8345697,NaN
9,10004_529.0_150.0_162.0_191.0_-16.0,1454318,NaN


**Bulgu:** Satırların %37.7'sinde `NaN` — bu, her `uid`'in ilk işlemi için beklenen davranış (`uid.nunique()` ile kabaca tutarlı). Kalanlarda medyan ~161,085 saniye (~1.9 gün), minimum **0 saniye** — aynı saniyeye denk gelen art arda işlemler var (otomatik/toplu işlem ya da şüpheli hızlı ardışıklık olabilir, modelleme aşamasında ayrıca incelenmeye değer).

## 8. Proxy Merchant Riski (`merchant_risk`)

Veri setinde açık bir `merchant_id` yok; en yakın proxy olarak `ProductCD + P_emaildomain` kombinasyonu kullanılıyor. Bu özellik `isFraud` etiketini kullandığı için diğerlerinden farklı: SADECE etiketli eğitim verisinde çalışır, canlı bir sistemde eğitim setinden üretilip dondurulmuş (frozen) bir lookup tablosu olarak servis edilmesi gerekir.

Leakage-safe Bayesian smoothing: her `merchant_id`'nin zamana göre kümülatif fraud toplamı/sayısı, veri setinin zamana göre kümülatif genel fraud oranına (`global_expanding_fraud_rate`) doğru çekiliyor (`m=100` ağırlıkla) — az örnekli merchant grupları aşırı öğrenmeye (overfitting) karşı korunuyor.

In [13]:
df = add_merchant_risk(df)
df[["merchant_id", "TransactionDT", "isFraud", "global_expanding_fraud_rate", "merchant_risk"]].head(20)

,merchant_id,TransactionDT,isFraud,global_expanding_fraud_rate,merchant_risk
292455,C_aim.com,9051270,0,0.033757,0.033757
292456,C_aim.com,10124733,0,0.034886,0.034541
171708,C_aim.com,10329773,0,0.035111,0.034422
138227,C_aim.com,15730122,0,0.034948,0.033930
105152,C_anonymous.com,87317,0,0.000000,0.000000
419360,C_anonymous.com,87928,0,0.000000,0.000000
203898,C_anonymous.com,87935,0,0.000000,0.000000
203900,C_anonymous.com,88404,0,0.000000,0.000000
203901,C_anonymous.com,88471,0,0.000000,0.000000
203902,C_anonymous.com,88777,0,0.000000,0.000000


**Bulgu:** İlk görülen satırlarda `merchant_risk` tam olarak `global_expanding_fraud_rate`'e eşit çıkıyor (grup geçmişi yokken smoothing tamamen global orana çekiliyor) — elle doğrulandı (örn. ikinci görülüşte `(0 + 100×0.034886)/(1+100) = 0.034541`, çıktıyla birebir eşleşiyor).

196 benzersiz proxy-merchant grubu (`ProductCD + P_emaildomain`) var. `merchant_risk` ile `isFraud` arasındaki korelasyon **0.20** — genel %3.5 fraud oranına göre anlamlı bir sinyal. Sadece 1 satırda (`count=590,539` / 590,540) `NaN` var — veri setinin en baştaki işlemi, docstring'de belirtilen kabul edilebilir istisna.

## Durum

Tamamlanan özellikler: `uid` (pseudo-kullanıcı kimliği), `avg_transaction_amount`, `amount_deviation_from_user`, `transactions_last_10min`, `transactions_last_24h`, `new_device`, `new_location`, `dist1_deviation_from_user`, `time_since_last_transaction`, `merchant_risk`.

Proje kapsamındaki tüm özellikler tamamlandı. `failed_attempts_last_hour` veri setinde mevcut olmadığı için kapsam dışı bırakıldı.